<a href="https://colab.research.google.com/github/suissai/whatsmeow-geoloc-webhook/blob/main/laya_legal_intent_finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Laya para classificação de intenção jurídica — Google Colab

Este notebook treina o Laya (`convaiinnovations/laya`) para classificar as 12 categorias do dataset jurídico de `suissa/intent-datasets-4finetuning`.

Pipeline:

`pedido humano → Laya → categoria jurídica → BehaviorID → Skill`

O notebook foi adaptado para **1 GPU no Google Colab**, usando micro-batch pequeno, gradient accumulation, gradient checkpointing, fp16, RLCD e calibração de confiança.

> O dataset atual é um piloto sintético: 96 exemplos de treino e 24 de avaliação. O objetivo aqui é validar o pipeline e medir confiança, não produzir um modelo jurídico pronto para produção.


## 0. Configuração

Se o Colab oferecer T4/L4, selecione GPU em `Runtime → Change runtime type → GPU`. O notebook continua funcionando em GPU única, mas uma GPU com mais VRAM será mais confortável.


In [1]:
!nvidia-smi

import os, sys, json, math, random, time, shutil, subprocess
import numpy as np
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}")
    print(f"VRAM: {p.total_memory / 1024**3:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    raise RuntimeError("Ative uma GPU no Colab antes de continuar.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


Sun Oct  4 19:29:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Instalar Laya e dependências


In [2]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy scikit-learn accelerate tabulate

import laya, transformers, datasets
print("Laya:", laya.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.0/323.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 87.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not c

## 2. Baixar o dataset jurídico e convertê-lo para o formato de typed decisions do Laya


In [5]:
# ============================================================
# 1. Baixar dataset e converter para o formato Laya
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO = Path("/content/intent-datasets-4finetuning")

# Limpa uma execução anterior incompleta
if REPO.exists():
    import shutil
    shutil.rmtree(REPO)

# Clone
result = subprocess.run(
    [
        "git",
        "clone",
        "--depth", "1",
        "https://github.com/suissa/intent-datasets-4finetuning.git",
        str(REPO),
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Git clone falhou com código {result.returncode}"
    )

# Confirma que o repositório realmente existe
if not (REPO / "scripts" / "legal_to_laya.py").exists():
    raise FileNotFoundError(
        f"Repositório incompleto: {REPO}"
    )

print("Repository:", REPO)
print("Converter:", REPO / "scripts" / "legal_to_laya.py")

# Converte train
subprocess.run(
    [
        sys.executable,
        "scripts/legal_to_laya.py",
        "--input",
        "datasets/legal/train.jsonl",
        "--output",
        "datasets/legal/laya/train.jsonl",
    ],
    cwd=REPO,
    check=True,
)

# Converte eval
subprocess.run(
    [
        sys.executable,
        "scripts/legal_to_laya.py",
        "--input",
        "datasets/legal/eval.jsonl",
        "--output",
        "datasets/legal/laya/eval.jsonl",
    ],
    cwd=REPO,
    check=True,
)

# Verificação
train_laya = REPO / "datasets/legal/laya/train.jsonl"
eval_laya = REPO / "datasets/legal/laya/eval.jsonl"

print()
print("TRAIN:", train_laya)
print("EVAL :", eval_laya)
print()
print("Primeiro exemplo:")
print(train_laya.read_text(encoding="utf-8").splitlines()[0][:2500])



Cloning into '/content/intent-datasets-4finetuning'...

Repository: /content/intent-datasets-4finetuning
Converter: /content/intent-datasets-4finetuning/scripts/legal_to_laya.py

TRAIN: /content/intent-datasets-4finetuning/datasets/legal/laya/train.jsonl
EVAL : /content/intent-datasets-4finetuning/datasets/legal/laya/eval.jsonl

Primeiro exemplo:
{"state": {"text": "Preciso entender uma cláusula do meu contrato.", "domain": "legal"}, "questions": {"intent": {"type": "choice", "instructions": "Qual categoria jurídica representa melhor a intenção do usuário?", "criteria": {"AUDIENCIA": "audiências, preparação e participação em audiência", "CONSULTA_JURIDICA": "orientação jurídica geral sobre direitos e situação", "CONSUMIDOR": "produtos, serviços, cobranças e direitos do consumidor", "CONTRATO": "contratos, cláusulas, rescisão e revisão contratual", "CRIMINAL": "acusações criminais, investigação, defesa e polícia", "EMPRESARIAL": "empresas, sócios, negócios e questões empresariais", "FA

## 3. Preparar o checkpoint base do Laya

O checkpoint público contém o encoder, tokenizer, pesos e `rl_agent_config.json`.


In [6]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config

MODEL_ID = "convaiinnovations/laya"
MODEL_DIR = snapshot_download(MODEL_ID)
_fix_tokenizer_config(MODEL_DIR)

with open(os.path.join(MODEL_DIR, "rl_agent_config.json")) as f:
    BASE_CFG = json.load(f)

tok = AutoTokenizer.from_pretrained(os.path.join(MODEL_DIR, "tokenizer"))
print("Checkpoint:", MODEL_DIR)
print("max_len base:", BASE_CFG.get("max_len"))
print("head_max_len base:", BASE_CFG.get("head_max_len"))
print("pad token:", tok.pad_token_id)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 39 files:   0%|          | 0/39 [00:00<?, ?it/s]

Checkpoint: /root/.cache/huggingface/hub/models--convaiinnovations--laya/snapshots/7b928d828b7b0e022f929d9bd2e44165aa270148
max_len base: 512
head_max_len base: 192
pad token: 50283


## 4. Pré-processamento

Cada utterance jurídica vira uma única pergunta `choice` com as 12 categorias. O alvo inicial é one-hot. Isso é deliberado: com apenas 96 exemplos, primeiro validamos o caminho de fine-tuning; depois podemos substituir os alvos por distribuições produzidas por um professor ou por anotação humana.


In [8]:
from laya.common import build_sequence, render_options, QTYPES
import json

TRAIN_JSONL = "/content/intent-datasets-4finetuning/datasets/legal/laya/train.jsonl"
EVAL_JSONL = "/content/intent-datasets-4finetuning/datasets/legal/laya/eval.jsonl"

MAX_LEN = 512
HEAD_MAX_LEN = 192

with open(TRAIN_JSONL, encoding="utf-8") as f:
    train_rows = [json.loads(line) for line in f if line.strip()]

with open(EVAL_JSONL, encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f if line.strip()]

print("Train rows:", len(train_rows))
print("Eval rows:", len(eval_rows))

# O formato Laya guarda a label em gold.intent.label
TRAIN_LABELS = {
    r["gold"]["intent"]["label"]
    for r in train_rows
}

EVAL_LABELS = {
    r["gold"]["intent"]["label"]
    for r in eval_rows
}

LABELS = sorted(TRAIN_LABELS | EVAL_LABELS)

print("Labels:", LABELS)
print("Número de opções:", len(LABELS))

QUESTIONS = {
    "legal_intent": {
        "type": "choice",
        "instructions": (
            "Qual categoria jurídica representa melhor "
            "a intenção principal do usuário?"
        ),
        "criteria": {
            label: label
            for label in LABELS
        },
    }
}

print("\nPrimeiro exemplo:")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2))

Train rows: 96
Eval rows: 24
Labels: ['AUDIENCIA', 'CONSULTA_JURIDICA', 'CONSUMIDOR', 'CONTRATO', 'CRIMINAL', 'EMPRESARIAL', 'FAMILIA', 'PETICAO', 'PRAZO', 'PREVIDENCIARIO', 'PROCESSO', 'TRABALHISTA']
Número de opções: 12

Primeiro exemplo:
{
  "state": {
    "text": "Preciso entender uma cláusula do meu contrato.",
    "domain": "legal"
  },
  "questions": {
    "intent": {
      "type": "choice",
      "instructions": "Qual categoria jurídica representa melhor a intenção do usuário?",
      "criteria": {
        "AUDIENCIA": "audiências, preparação e participação em audiência",
        "CONSULTA_JURIDICA": "orientação jurídica geral sobre direitos e situação",
        "CONSUMIDOR": "produtos, serviços, cobranças e direitos do consumidor",
        "CONTRATO": "contratos, cláusulas, rescisão e revisão contratual",
        "CRIMINAL": "acusações criminais, investigação, defesa e polícia",
        "EMPRESARIAL": "empresas, sócios, negócios e questões empresariais",
        "FAMILIA": "di

In [10]:
def make_item(row):
    state = row["state"]["text"]
    label = row["gold"]["intent"]["label"]
    q = QUESTIONS["legal_intent"]
    crit = q["criteria"]
    target = [1.0 if x == label else 0.0 for x in LABELS]
    seq, markers = build_sequence(
        tok,
        state,
        {"t": "choice", "ins": q["instructions"], "crit": crit},
        MAX_LEN,
        HEAD_MAX_LEN,
    )
    expected_k = len(render_options({"t": "choice", "crit": crit}))
    if len(markers) != expected_k:
        raise ValueError(f"Option markers mismatch: {len(markers)} != {expected_k}")
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES["choice"],
        "target": target,
        "label": LABELS.index(label),
        "text": state,
        "gold_label": label,
    }

train_items_all = [make_item(r) for r in train_rows]
eval_items = [make_item(r) for r in eval_rows]

print("Preprocessed train:", len(train_items_all))
print("Preprocessed eval:", len(eval_items))
print("Example:", train_items_all[0]["gold_label"], train_items_all[0]["text"])

# Keep 20% of train as a small calibration set. It is never used for gradient updates.
rng = random.Random(20261004)
idx = list(range(len(train_items_all)))
rng.shuffle(idx)
n_calib = max(12, int(round(len(idx) * 0.20)))
calib_items = [train_items_all[i] for i in idx[:n_calib]]
train_items = [train_items_all[i] for i in idx[n_calib:]]

print(f"Gradient-train items: {len(train_items)}")
print(f"Calibration items: {len(calib_items)}")

Preprocessed train: 96
Preprocessed eval: 24
Example: CONTRATO Preciso entender uma cláusula do meu contrato.
Gradient-train items: 77
Calibration items: 19


## 5. Treinamento RLCD em uma GPU

Configuração conservadora para Colab/T4:

- 2 épocas
- micro-batch 1
- gradient accumulation 32
- `max_len=512`
- `head_max_len=192`
- fp16
- gradient checkpointing
- encoder LR `2.5e-5`
- head LR `1e-4`
- ruído RLCD `0.4 → 0.1`

O effective batch é 32 sequências. Como o dataset é pequeno, aumentar muito o número de épocas pode memorizar rapidamente.


In [11]:
OUTPUT_DIR = "/content/laya_legal_finetuned"
TRAIN_ITEMS_PATH = "/content/legal_train_items.pt"
CALIB_ITEMS_PATH = "/content/legal_calib_items.pt"

# Persist CPU-side preprocessed items so the training code can be rerun without tokenizing again.
torch.save(train_items, TRAIN_ITEMS_PATH)
torch.save(calib_items, CALIB_ITEMS_PATH)

# Write a single-GPU training program. It follows the public Laya RLCD recipe,
# but removes DDP because standard Colab normally exposes one GPU.
train_script = r"""
import os, sys, time, json, random, math
import torch
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward


def collate(items, pad_id):
    n = len(items)
    L = max(len(x["ids"]) for x in items)
    K = max(len(x["markers"]) for x in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, K), dtype=torch.long)
    mmask = torch.zeros((n, K), dtype=torch.bool)
    target = torch.zeros((n, K), dtype=torch.float32)
    qtype = torch.tensor([x["qtype"] for x in items], dtype=torch.long)
    labels = torch.tensor([x["label"] for x in items], dtype=torch.long)
    for i, x in enumerate(items):
        ids[i, :len(x["ids"])] = torch.tensor(x["ids"], dtype=torch.long)
        att[i, :len(x["ids"])] = 1
        k = len(x["markers"])
        mpos[i, :k] = torch.tensor(x["markers"], dtype=torch.long)
        mmask[i, :k] = True
        target[i, :len(x["target"])] = torch.tensor(x["target"], dtype=torch.float32)
    return ids, att, mpos, mmask, target, qtype, labels


def run():
    model_dir, train_path, out_dir = sys.argv[1:4]
    epochs = 2
    micro_batch = 1
    grad_accum = 32
    group_size = 4
    lr_encoder = 2.5e-5
    lr_head = 1e-4
    sigma_start, sigma_end = 0.4, 0.1

    device = torch.device("cuda")
    os.makedirs(out_dir, exist_ok=True)

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["max_len"] = 512
    cfg["head_max_len"] = 192
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    items = torch.load(train_path, weights_only=False)
    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": lr_encoder},
        {"params": head_params, "lr": lr_head},
    ], weight_decay=0.01)
    updates_per_epoch = max(1, math.ceil(len(items) / micro_batch / grad_accum))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, updates_per_epoch * epochs), eta_min=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    print(f"Training {len(items)} items | epochs={epochs} | micro={micro_batch} | accum={grad_accum}")
    t0 = time.time()
    for epoch in range(epochs):
        random.Random(1000 + epoch).shuffle(items)
        progress = epoch / max(1, epochs - 1)
        sigma = sigma_start + (sigma_end - sigma_start) * progress
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0
        steps = 0
        for bidx in range(0, len(items), micro_batch):
            chunk = items[bidx:bidx + micro_batch]
            ids, att, mpos, mmask, target, qtype, labels = collate(chunk, tok.pad_token_id)
            ids, att, mpos, mmask = [x.to(device) for x in (ids, att, mpos, mmask)]
            target, qtype = target.to(device), qtype.to(device)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = model(ids, att, mpos, mmask, qtype)
            logits = logits.float()
            k = mmask.sum(-1, keepdim=True).float()
            eps = torch.randn((group_size,) + logits.shape, device=device) * sigma * mmask
            eps = (eps - eps.sum(-1, keepdim=True) / k.clamp_min(1.0)) * mmask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mmask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), qtype, mmask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mmask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mmask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + loss_ce) / grad_accum + 0.0 * act.sum()
            scaler.scale(loss).backward()
            steps += 1
            epoch_loss += float(loss.detach().cpu()) * grad_accum

            if steps % grad_accum == 0 or bidx + micro_batch >= len(items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        print(f"Epoch {epoch+1}/{epochs}: avg_loss={epoch_loss/max(1,steps):.5f} sigma={sigma:.3f} elapsed={time.time()-t0:.1f}s")
        ckpt = os.path.join(out_dir, "checkpoint_latest")
        os.makedirs(ckpt, exist_ok=True)
        save_file({k:v.detach().half().cpu().contiguous() for k,v in model.state_dict().items()}, os.path.join(ckpt, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(ckpt, "encoder"))
        tok.save_pretrained(os.path.join(ckpt, "tokenizer"))

    # Final model artifacts.
    save_file({k:v.detach().half().cpu().contiguous() for k,v in model.state_dict().items()}, os.path.join(out_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(out_dir, "encoder"))
    tok.save_pretrained(os.path.join(out_dir, "tokenizer"))
    cfg["max_len"] = 512
    cfg["head_max_len"] = 192
    cfg.pop("temperature_by_options", None)
    with open(os.path.join(out_dir, "rl_agent_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    print("Saved:", out_dir)

if __name__ == "__main__":
    run()
"""

with open("/content/train_laya_legal.py", "w") as f:
    f.write(train_script)
print("Training script written")


Training script written


## 6. Executar o fine-tuning

Esta etapa pode levar alguns minutos em uma T4/L4. Se ocorrer OOM, reduza `MAX_LEN` para 384 e `HEAD_MAX_LEN` para 160 na célula de pré-processamento e reinicie a sessão.


In [12]:
!python /content/train_laya_legal.py "{MODEL_DIR}" "{TRAIN_ITEMS_PATH}" "{OUTPUT_DIR}"


Training 77 items | epochs=2 | micro=1 | accum=32
/content/train_laya_legal.py:113: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch 1/2: avg_loss=2.27489 sigma=0.400 elapsed=18.4s
Epoch 2/2: avg_loss=1.39841 sigma=0.100 elapsed=36.9s
Saved: /content/laya_legal_finetuned


## 7. Avaliação no conjunto jurídico separado

A avaliação usa os 24 exemplos de `datasets/legal/eval.jsonl`, que não participam do gradiente. Além de accuracy e macro-F1, medimos a confiança da classe vencedora.


In [14]:
import laya
import time
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

agent = laya.Agent(OUTPUT_DIR, device="cuda")

pred_labels = []
gold_labels = []
confidences = []
rows_eval = []
latencies_ms = []

for row in eval_rows:
    text = row["state"]["text"]
    label = row["gold"]["intent"]["label"]
    t0 = time.perf_counter()
    result = agent.predict(text, QUESTIONS)
    dt = (time.perf_counter() - t0) * 1000
    ans = result["answers"]["legal_intent"]
    probs = ans["probabilities"]
    pred = ans["choice"]
    conf = float(ans.get("answer_confidence", max(probs.values())))
    pred_labels.append(pred)
    gold_labels.append(label)
    confidences.append(conf)
    latencies_ms.append(dt)
    rows_eval.append({
        "text": text,
        "gold": label,
        "pred": pred,
        "confidence": conf,
        "probabilities": probs,
        "correct": pred == label,
        "latency_ms": dt,
    })

accuracy = accuracy_score(gold_labels, pred_labels)
macro_f1 = f1_score(gold_labels, pred_labels, labels=LABELS, average="macro", zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro-F1 : {macro_f1:.4f}")
print(f"Latency p50: {np.percentile(latencies_ms,50):.2f} ms")
print(f"Latency p95: {np.percentile(latencies_ms,95):.2f} ms")
print("\nClassification report:\n")
print(classification_report(gold_labels, pred_labels, labels=LABELS, zero_division=0))

Accuracy : 0.6250
Macro-F1 : 0.6065
Latency p50: 33.97 ms
Latency p95: 40.95 ms

Classification report:

                   precision    recall  f1-score   support

        AUDIENCIA       1.00      1.00      1.00         2
CONSULTA_JURIDICA       0.29      1.00      0.44         2
       CONSUMIDOR       0.50      0.50      0.50         2
         CONTRATO       1.00      1.00      1.00         2
         CRIMINAL       1.00      1.00      1.00         2
      EMPRESARIAL       1.00      0.50      0.67         2
          FAMILIA       0.00      0.00      0.00         2
          PETICAO       1.00      1.00      1.00         2
            PRAZO       1.00      0.50      0.67         2
   PREVIDENCIARIO       0.00      0.00      0.00         2
         PROCESSO       1.00      1.00      1.00         2
      TRABALHISTA       0.00      0.00      0.00         2

         accuracy                           0.62        24
        macro avg       0.65      0.62      0.61        24
     wei

## 8. Medir calibração: Brier, ECE e confiança em acertos/erros

A métrica mais importante para o seu executor não é somente acertar a categoria. Queremos saber se `confidence=0.95` realmente significa algo próximo de 95% de acerto.


In [15]:
def prob_vector(item):
    return np.array([float(item["probabilities"].get(label, 0.0)) for label in LABELS], dtype=np.float64)

y_true = np.array([LABELS.index(x) for x in gold_labels])
P = np.vstack([prob_vector(x) for x in rows_eval])
P = P / np.clip(P.sum(axis=1, keepdims=True), 1e-12, None)
y_pred = P.argmax(axis=1)
conf = P.max(axis=1)
correct = (y_pred == y_true).astype(int)

# Multiclass Brier score.
onehot = np.zeros_like(P)
onehot[np.arange(len(y_true)), y_true] = 1.0
brier = np.mean(np.sum((P - onehot) ** 2, axis=1))

def ece(conf, correct, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    details = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf >= lo) & ((conf < hi) if hi < 1 else (conf <= hi))
        if not mask.any():
            continue
        acc = correct[mask].mean()
        avg_conf = conf[mask].mean()
        weight = mask.mean()
        total += weight * abs(acc - avg_conf)
        details.append((lo, hi, int(mask.sum()), float(avg_conf), float(acc)))
    return float(total), details

ece_value, ece_bins = ece(conf, correct)

print(f"Brier score: {brier:.4f}")
print(f"ECE (10 bins): {ece_value:.4f}")
print(f"Mean confidence: {conf.mean():.4f}")
print(f"Mean confidence — correct: {conf[correct==1].mean() if (correct==1).any() else float('nan'):.4f}")
print(f"Mean confidence — wrong:   {conf[correct==0].mean() if (correct==0).any() else float('nan'):.4f}")

print("\nECE bins: [low, high, n, mean_conf, accuracy]")
for x in ece_bins:
    print(x)


Brier score: 0.4572
ECE (10 bins): 0.1909
Mean confidence: 0.6123
Mean confidence — correct: 0.7588
Mean confidence — wrong:   0.3682

ECE bins: [low, high, n, mean_conf, accuracy]
(np.float64(0.1), np.float64(0.2), 3, 0.1402193114021931, 0.0)
(np.float64(0.2), np.float64(0.30000000000000004), 2, 0.2208603954567536, 0.5)
(np.float64(0.30000000000000004), np.float64(0.4), 1, 0.30146985301469853, 1.0)
(np.float64(0.4), np.float64(0.5), 2, 0.42009200920092005, 0.0)
(np.float64(0.5), np.float64(0.6000000000000001), 4, 0.5382581322439384, 0.5)
(np.float64(0.6000000000000001), np.float64(0.7000000000000001), 2, 0.6960999999999999, 1.0)
(np.float64(0.7000000000000001), np.float64(0.8), 1, 0.7248275172482752, 0.0)
(np.float64(0.8), np.float64(0.9), 3, 0.871366445907331, 1.0)
(np.float64(0.9), np.float64(1.0), 6, 0.9677843980485094, 1.0)


## 9. O teste de confiança que interessa para o executor

Aqui medimos quantos exemplos seriam aceitos se o executor exigisse uma confiança mínima. Isso permite definir uma política como:

`confidence >= 0.90 → executa automaticamente`

`confidence < 0.90 → pergunta ao humano / Human-in-the-Healing-Loop`

Os números abaixo são apenas os números deste conjunto de avaliação; não são uma garantia de produção.


In [16]:
thresholds = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.98]
threshold_rows = []
for t in thresholds:
    mask = conf >= t
    coverage = float(mask.mean())
    conditional_acc = float(correct[mask].mean()) if mask.any() else float("nan")
    threshold_rows.append({
        "threshold": t,
        "coverage": coverage,
        "accepted": int(mask.sum()),
        "accuracy_when_accepted": conditional_acc,
    })

import pandas as pd
threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.to_string(index=False))


 threshold  coverage  accepted  accuracy_when_accepted
      0.50  0.666667        16                0.812500
      0.60  0.500000        12                0.916667
      0.70  0.416667        10                0.900000
      0.80  0.375000         9                1.000000
      0.85  0.333333         8                1.000000
      0.90  0.250000         6                1.000000
      0.95  0.250000         6                1.000000
      0.98  0.041667         1                1.000000


## 10. Matriz de confusão

Em classificação jurídica, os erros mais úteis são os pares de categorias que o modelo confunde. Eles viram candidatos a hard negatives para a próxima versão do dataset.


In [17]:
cm = confusion_matrix(gold_labels, pred_labels, labels=LABELS)
cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
print(cm_df.to_string())

print("\nErros:")
for x in rows_eval:
    if not x["correct"]:
        print(f"[{x['confidence']:.3f}] {x['gold']} -> {x['pred']} | {x['text']}")


                   AUDIENCIA  CONSULTA_JURIDICA  CONSUMIDOR  CONTRATO  CRIMINAL  EMPRESARIAL  FAMILIA  PETICAO  PRAZO  PREVIDENCIARIO  PROCESSO  TRABALHISTA
AUDIENCIA                  2                  0           0         0         0            0        0        0      0               0         0            0
CONSULTA_JURIDICA          0                  2           0         0         0            0        0        0      0               0         0            0
CONSUMIDOR                 0                  0           1         0         0            0        0        0      0               0         0            1
CONTRATO                   0                  0           0         2         0            0        0        0      0               0         0            0
CRIMINAL                   0                  0           0         0         2            0        0        0      0               0         0            0
EMPRESARIAL                0                  1           

## 11. Exemplos mais incertos

Esses casos são especialmente úteis para criar novos exemplos de treino, porque mostram onde a decisão ainda não é suficientemente separável.


In [18]:
for x in sorted(rows_eval, key=lambda r: r["confidence"])[:10]:
    print(f"confidence={x['confidence']:.4f} | gold={x['gold']} | pred={x['pred']}")
    print(" ", x["text"])
    print()


confidence=0.1125 | gold=FAMILIA | pred=TRABALHISTA
  Quero saber como funciona um divórcio.

confidence=0.1507 | gold=FAMILIA | pred=CONSULTA_JURIDICA
  Quero fazer um acordo de divórcio.

confidence=0.1575 | gold=TRABALHISTA | pred=CONSULTA_JURIDICA
  Quero saber se tenho direito às férias não pagas.

confidence=0.2165 | gold=CONSUMIDOR | pred=TRABALHISTA
  Quero resolver um produto que veio com defeito.

confidence=0.2252 | gold=CONSULTA_JURIDICA | pred=CONSULTA_JURIDICA
  Quero saber quais são meus direitos neste caso.

confidence=0.3015 | gold=PROCESSO | pred=PROCESSO
  Quero verificar se houve uma decisão recente.

confidence=0.4131 | gold=TRABALHISTA | pred=CONSUMIDOR
  Quero cobrar adicional noturno.

confidence=0.4270 | gold=PREVIDENCIARIO | pred=CONSULTA_JURIDICA
  Quero saber se tenho direito à aposentadoria.

confidence=0.5132 | gold=CONSULTA_JURIDICA | pred=CONSULTA_JURIDICA
  Quero obter uma análise jurídica da minha situação.

confidence=0.5160 | gold=PRAZO | pred=CONSUL

## 12. Teste manual com intenções novas

Este bloco não entra no cálculo da métrica. Ele serve para observar a distribuição completa e testar frases que não estão no `eval.jsonl`.


In [19]:
manual_tests = [
    "Quero saber quanto tempo ainda tenho para recorrer de uma decisão.",
    "Preciso preparar uma petição para apresentar no processo.",
    "Quero saber como está o andamento da minha ação.",
    "Meu produto chegou quebrado e a loja não quer trocar.",
    "Fui demitido e quero saber quais verbas devo receber.",
    "Quero revisar o contrato antes de assinar.",
    "Quero saber se tenho direito à aposentadoria.",
    "Preciso de ajuda com um divórcio.",
]

for text in manual_tests:
    result = agent.predict(text, QUESTIONS)
    ans = result["answers"]["legal_intent"]
    print("\n", text)
    print("pred:", ans["choice"])
    print("confidence:", ans.get("answer_confidence"))
    print("probabilities:")
    for label, p in sorted(ans["probabilities"].items(), key=lambda kv: kv[1], reverse=True):
        print(f"  {label:22s} {p:.4f}")



 Quero saber quanto tempo ainda tenho para recorrer de uma decisão.
pred: CONSULTA_JURIDICA
confidence: 0.2634
probabilities:
  CONSULTA_JURIDICA      0.2634
  CRIMINAL               0.2252
  PROCESSO               0.1687
  TRABALHISTA            0.0687
  EMPRESARIAL            0.0564
  PREVIDENCIARIO         0.0485
  PRAZO                  0.0419
  CONTRATO               0.0291
  CONSUMIDOR             0.0275
  AUDIENCIA              0.0239
  PETICAO                0.0235
  FAMILIA                0.0232

 Preciso preparar uma petição para apresentar no processo.
pred: PETICAO
confidence: 0.9621
probabilities:
  PETICAO                0.9621
  PROCESSO               0.0197
  CONSULTA_JURIDICA      0.0021
  AUDIENCIA              0.0019
  EMPRESARIAL            0.0019
  PREVIDENCIARIO         0.0019
  CONTRATO               0.0018
  PRAZO                  0.0018
  TRABALHISTA            0.0018
  CONSUMIDOR             0.0017
  CRIMINAL               0.0017
  FAMILIA                0.00

## 13. Gerar benchmark JSON

O arquivo pode ser usado depois para comparar diferentes versões do dataset/modelo.


In [20]:
report = {
    "task": "legal_intent_classification",
    "model": "Laya fine-tuned",
    "base_model": MODEL_ID,
    "dataset": "suissa/intent-datasets-4finetuning",
    "train_examples": len(train_rows),
    "calibration_examples": len(calib_items),
    "eval_examples": len(eval_rows),
    "labels": LABELS,
    "metrics": {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "brier": float(brier),
        "ece": float(ece_value),
        "mean_confidence": float(conf.mean()),
        "mean_confidence_correct": float(conf[correct==1].mean()) if (correct==1).any() else None,
        "mean_confidence_wrong": float(conf[correct==0].mean()) if (correct==0).any() else None,
        "latency_p50_ms": float(np.percentile(latencies_ms,50)),
        "latency_p95_ms": float(np.percentile(latencies_ms,95)),
    },
    "confidence_thresholds": threshold_rows,
    "confusion_matrix": cm_df.to_dict(),
    "predictions": rows_eval,
}

REPORT_PATH = "/content/laya_legal_benchmark.json"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Saved:", REPORT_PATH)
print(json.dumps(report["metrics"], indent=2, ensure_ascii=False))


Saved: /content/laya_legal_benchmark.json
{
  "accuracy": 0.625,
  "macro_f1": 0.6064814814814815,
  "brier": 0.45721736671305724,
  "ece": 0.19093773453713306,
  "mean_confidence": 0.6122874320322377,
  "mean_confidence_correct": 0.7587502040853015,
  "mean_confidence_wrong": 0.3681828119437978,
  "latency_p50_ms": 33.970206000049075,
  "latency_p95_ms": 40.94654359978449
}


## 14. Salvar no Google Drive (opcional)

Execute esta célula somente se quiser preservar o checkpoint e o benchmark depois que a sessão do Colab terminar.


In [21]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/laya-legal-finetuned"
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree(OUTPUT_DIR, os.path.join(DRIVE_DIR, "checkpoint"), dirs_exist_ok=True)
shutil.copy(REPORT_PATH, os.path.join(DRIVE_DIR, "laya_legal_benchmark.json"))
print("Saved to:", DRIVE_DIR)


Mounted at /content/drive
Saved to: /content/drive/MyDrive/laya-legal-finetuned


## 15. Downloadar o checkpoint como `.tar.gz`

Use esta célula para gerar um único arquivo que pode ser baixado do Colab.


In [22]:
ARCHIVE = "/content/laya_legal_finetuned.tar.gz"
!tar -czf "{ARCHIVE}" -C /content "$(basename "{OUTPUT_DIR}")"
print(ARCHIVE)

from google.colab import files
files.download(ARCHIVE)


/content/laya_legal_finetuned.tar.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 16. Interpretação do resultado

Para o seu executor, não use apenas `argmax`. O resultado deve ser tratado como uma decisão com distribuição e confiança:

```text
Human Intent
    ↓
Laya
    ↓
{ category, probabilities, confidence }
    ↓
confidence alta → BehaviorID → Skill
confidence baixa → pergunta / Human-in-the-Healing-Loop
```

O próximo passo científico é aumentar o dataset com hard negatives, especialmente entre categorias semanticamente próximas, e então repetir o experimento mantendo este mesmo `eval.jsonl` congelado.
